# Module 4: Annotation Queues

> Self-directed module, ~20 min.

Picking up from Module 3: the online evaluator is scoring new runs as they land. This module wires the next link in the chain — annotation queues collect the runs the auto-eval flagged as failing, so a human reviewer can label them. Those labels become the training data for the next iteration of the judge, the agent, or a dataset.

Workflow: create a queue → register a routing rule that listens for failing feedback → generate traces designed to fail → review and label in the UI.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)


## Step 1 — Create the queue

Queues are named buckets that hold runs awaiting human review. `get_or_create_annotation_queue` is idempotent — re-running this notebook reuses the existing queue rather than producing duplicates. Names are project-scoped, so a single workspace can hold many queues for different review workflows.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="client-research-needs-review",
    description="Client research runs flagged by the online correctness rule (Module 3) as failing.",
)
print(f"Queue: {queue.name} (id={queue.id})")


## Step 2 — Route flagged runs to the queue

The routing rule reuses the same `create_run_rule` API as Module 3, but with `add_to_annotation_queue_id` instead of a judge prompt — same machinery, different effect. The filter is what makes it route only the failures:

- `eq(is_root, true)` — only top-level runs (skip child spans)
- `eq(feedback_key, "correctness")` — only runs the online evaluator has scored
- `lt(feedback_score, 0.5)` — only failures


In [ ]:
queue_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="client-research-route-failures",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'lt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID: ", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


Runs won't appear in the queue immediately — the feedback from Module 3 has to land first (~30 seconds after each run completes), and only failing runs get routed. If the queue is empty after Step 3, give it another half-minute.


### Caveat: a well-behaved agent may produce an empty queue

The client research agent in this POC is intentionally simple — three tools, one subagent, a clear workflow. On most prompts the online judge will score `correctness=true`, and nothing routes to the queue. That's correct behavior, not a bug, but it does make the queue UI hard to demo.

If the queue ends up empty after Step 3:

- **Raise the threshold:** change `lt(feedback_score, 0.5)` to `lt(feedback_score, 1.0)` and re-create the rule. Routes everything — useful for demos, not for production.
- **Tighten the judge** (in Module 3's rule, editable in the UI): stricter criteria → more failures → more queue entries.
- **Push runs manually:** `client.add_runs_to_annotation_queue(queue_id=queue.id, run_ids=[...])` bypasses the rule entirely. Useful when reviewing specific runs by ID.


## Step 3 — Generate traces that should fail

These prompts are designed to invite the failure modes the judge looks for: unknown clients, unanswerable specifics, future predictions, and one-word answers without justification. Real POCs would tune the judge against actual failure modes observed in production.


In [ ]:
trigger_prompts = [
    "Look up the client 'globex-corp' and tell me about their holdings.",
    "What was Smith Family Office's exact return last quarter?",
    "How will AAPL stock perform next month?",
    "Give me a one-word answer: should I rebalance Acme Pension Fund?",
]

for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")


## Step 4 — Review in the UI

The deep link below opens the queue's review pane. Each entry shows the run's input, the agent's response, and the full trace tree side-by-side with labeling controls. Reviewer actions include accept, reject, free-text comments, and structured labels (configurable per queue). Keyboard shortcuts speed up bulk review — `j` / `k` to navigate, number keys for scoring.

<img src="../images/04-queue-review.png" alt="Annotation queue review pane with a flagged run open" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
tenant_id = queue_rule["payload"]["tenant_id"]
queue_url = f"https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}"
print(f"Queue (review UI): {queue_url}")
print(f"Routing rule:      {queue_rule['url']}")


## Cleanup

Leave the queue and rule active if iterating. To tear them down:

```python
delete_run_rule(client, queue_rule["id"])
client.delete_annotation_queue(queue_id=queue.id)
```


## Recap

| Topic | API |
|---|---|
| Create queue | `get_or_create_annotation_queue(client, name=..., description=...)` |
| Route runs to queue | `create_run_rule(..., filter=..., add_to_annotation_queue_id=queue.id)` |
| Route failures only | filter ends with `lt(feedback_score, 0.5)` |
| Manual push | `client.add_runs_to_annotation_queue(queue_id=..., run_ids=[...])` |

Next: `05_datasets_and_experiments.ipynb` — promote labeled runs into a dataset and run offline experiments against it.
